In [1]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u

from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd
from reproject import reproject_interp

from spectral_cube import SpectralCube as sc

from tqdm.notebook import trange
import glob
import warnings
warnings.filterwarnings("ignore")

In [ ]:
fits_file_list = glob.glob('./*chop.fits')
hdus = [fits.open(f)[0] for f in fits_file_list]
n_chan = hdus[0].data.shape[0]
n_chan

1024

In [3]:
# 先为第0通道做WCS计算
slice_hdus = []
for hdu in hdus:
    data2d = hdu.data[0]
    wcs3d = WCS(hdu.header)
    wcs2d = wcs3d.celestial
    header2d = wcs2d.to_header()
    slice_hdus.append(fits.PrimaryHDU(data2d, header=header2d))
wcs_out, shape_out = find_optimal_celestial_wcs(slice_hdus, projection='CAR')

wcs_out.wcs.crpix[1] -= wcs_out.wcs.crval[1] / wcs_out.wcs.cdelt[1]
wcs_out.wcs.crval[1] = 0
wcs_out, shape_out

(WCS Keywords
 
 Number of WCS axes: 2
 CTYPE : 'RA---CAR' 'DEC--CAR' 
 CRVAL : 99.50374974942945 0.0 
 CRPIX : 679.7561336659564 259.3424159570403 
 PC1_1 PC1_2  : 1.0 0.0 
 PC2_1 PC2_2  : 0.0 1.0 
 CDELT : -0.0666666701436 0.0666666701436 
 NAXIS : 0  0,
 (309, 1360))

In [4]:
# 逐通道马赛克
mosaics = []
for i in trange(144):  # -600km/s to 600km/s --> channel 52 to 143 (count from 0)
    slice_hdus = []
    for hdu in hdus:
        data2d = hdu.data[i]
        wcs3d = WCS(hdu.header)
        wcs2d = wcs3d.celestial
        header2d = wcs2d.to_header()
        slice_hdus.append(fits.PrimaryHDU(data2d, header=header2d))
    array, footprint = reproject_and_coadd(
        slice_hdus,
        wcs_out,
        shape_out=shape_out,
        reproject_function=reproject_interp
    )
    mosaics.append(array)

mosaic_cube = np.stack(mosaics, axis=0)

  0%|          | 0/144 [00:00<?, ?it/s]

In [5]:
# 假设你有原始 cube 的 header 和新的 wcs_out
new_header = hdu.header.copy()  # hdu_cube 是你的原始cube HDU
header_out = wcs_out.to_header()        # wcs_out 是新的2D WCS对象

# 覆盖原header的第一、第二坐标轴相关关键字
for key in header_out:
    if key[-1] in ('1', '2'):  # 只替换结尾为1或2的关键字
        new_header[key] = header_out[key]

# new_header 现在就是你要用的新header

new_header['NAXIS3'] = 196
#new_header['CRPIX3'] = 22
#new_header['ALTRPIX'] = 22
del new_header['HISTORY']

In [6]:
wcs_out = WCS(new_header)
new_header

SIMPLE  =                    T /Standard FITS                                   
BITPIX  =                  -32 / Short integer (16 bit)                         
NAXIS   =                    3                                                  
NAXIS1  =                  170                                                  
NAXIS2  =                  160                                                  
NAXIS3  =                  196                                                  
DATAMIN =       -8.0000000E+00                                                  
DATAMAX =        3.2000000E+01                                                  
BMAJ    =        2.5833334E-01                                                  
BMIN    =        2.5833334E-01                                                  
BPA     =        0.0000000E+00                                                  
                                                                                
BUNIT   = 'JY/BEAM '        

In [7]:
hdu_mosaic = fits.PrimaryHDU(data=mosaic_cube, header=new_header)

In [8]:
hipass_cube = sc.read(hdu_mosaic)
hipass_cube = hipass_cube.subcube(
    xlo=60 * u.deg,
    xhi=140 * u.deg,
    ylo=-13 * u.deg,
    yhi=2 * u.deg,
    zlo=-600 * u.km / u.s,
    zhi=600 * u.km / u.s,
)
hipass_cube

SpectralCube with shape=(92, 226, 1201) and unit=Jy / beam:
 n_x:   1201  type_x: RA---CAR  unit_x: deg    range:    60.020823 deg:  140.020827 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     92  type_s: VRAD      unit_s: m / s  range:  -600308.555 m / s:  600101.682 m / s

In [9]:
# Find velocity axis (assume axis=2, adjust if necessary)
velocity_hel = hipass_cube.spectral_axis.to(u.km / u.s)
velocity_hel

<Quantity [-600.308555  , -587.11723371, -573.92591241, -560.73459112,
           -547.54326983, -534.35194854, -521.16062725, -507.96930596,
           -494.77798467, -481.58666338, -468.39534209, -455.20402079,
           -442.0126995 , -428.82137821, -415.63005692, -402.43873563,
           -389.24741434, -376.05609305, -362.86477176, -349.67345047,
           -336.48212918, -323.29080788, -310.09948659, -296.9081653 ,
           -283.71684401, -270.52552272, -257.33420143, -244.14288014,
           -230.95155885, -217.76023756, -204.56891626, -191.37759497,
           -178.18627368, -164.99495239, -151.8036311 , -138.61230981,
           -125.42098852, -112.22966723,  -99.03834594,  -85.84702465,
            -72.65570335,  -59.46438206,  -46.27306077,  -33.08173948,
            -19.89041819,   -6.6990969 ,    6.49222439,   19.68354568,
             32.87486697,   46.06618827,   59.25750956,   72.44883085,
             85.64015214,   98.83147343,  112.02279472,  125.21411601,
      

In [10]:
ra = 100 * u.deg
dec = 0 * u.deg
velocity_lsrk = []
for v in velocity_hel:
    skycoord = SkyCoord(ra, dec, radial_velocity=v, frame="fk5", pm_ra_cosdec=0*u.mas/u.yr, pm_dec=0*u.mas/u.yr, distance=10*u.kpc)
    velocity_lsrk.append(skycoord.lsrk.radial_velocity.value)
velocity_lsrk = np.array(velocity_lsrk) * u.km / u.s
velocity_lsrk

<Quantity [-617.41308836, -604.22176707, -591.03044578, -577.83912449,
           -564.6478032 , -551.4564819 , -538.26516061, -525.07383932,
           -511.88251803, -498.69119674, -485.49987545, -472.30855416,
           -459.11723287, -445.92591158, -432.73459029, -419.54326899,
           -406.3519477 , -393.16062641, -379.96930512, -366.77798383,
           -353.58666254, -340.39534125, -327.20401996, -314.01269867,
           -300.82137737, -287.63005608, -274.43873479, -261.2474135 ,
           -248.05609221, -234.86477092, -221.67344963, -208.48212834,
           -195.29080705, -182.09948576, -168.90816446, -155.71684317,
           -142.52552188, -129.33420059, -116.1428793 , -102.95155801,
            -89.76023672,  -76.56891543,  -63.37759414,  -50.18627284,
            -36.99495155,  -23.80363026,  -10.61230897,    2.57901232,
             15.77033361,   28.9616549 ,   42.15297619,   55.34429748,
             68.53561877,   81.72694007,   94.91826136,  108.10958265,
      

In [11]:
delta_v = []
for i in range(len(velocity_lsrk)):
    delta_v.append(velocity_lsrk[i].value - velocity_hel[i].value)
delta_v = np.mean(delta_v) * 1000
delta_v

np.float64(-17104.53336360639)

In [ ]:
# Now velocity_lsrk is your velocities in LSRK
cube_header = hipass_cube.header
cube_header['CRVAL3'] += delta_v

old_CRVAL1 = cube_header['CRVAL1']
new_CRVAL1 = 100
old_CRPIX1 = cube_header['CRPIX1']
CDELT1 = cube_header['CDELT1']
new_CRPIX1 = old_CRPIX1 - (old_CRVAL1 - new_CRVAL1) / CDELT1
cube_header['CRVAL1'] = new_CRVAL1
cube_header['CRPIX1'] = new_CRPIX1

# Update header
cube_header['SPECSYS'] = 'LSRK'
del cube_header['SLICE']
cube_header['BLANK'] = 0

# Convert Jy/beam to K using brightness temperature equivalency
freq = hipass_cube.with_spectral_unit(u.Hz).spectral_axis
hipass_cube_K = hipass_cube.to(u.K, equivalencies=u.brightness_temperature(freq))

cube_header['BUNIT'] = "K"

In [13]:
hipass_cube_K_hdu = fits.PrimaryHDU(data=hipass_cube_K.unmasked_data[:, :, :].value, header=cube_header)
hipass_cube = sc.read(hipass_cube_K_hdu)

In [ ]:
# Save to new FITS file
hipass_cube.write('./HIPASS_mosaic_cube.fits', overwrite=True)

hipass_cube_K_60_80_p = hipass_cube_K.subcube(xlo=60 * u.deg, xhi=80 * u.deg, zlo=83*u.km/u.s, zhi=283*u.km/u.s)
hipass_cube_K_60_80_p.write('./HIPASS_mosaic_cube_60_80_+.fits', overwrite=True)

hipass_cube_K_60_80_n = hipass_cube_K.subcube(xlo=60 * u.deg, xhi=80 * u.deg, zlo=-255*u.km/u.s, zhi=-55*u.km/u.s)
hipass_cube_K_60_80_n.write('./HIPASS_mosaic_cube_60_80_-.fits', overwrite=True)

hipass_cube_K_80_100_p = hipass_cube_K.subcube(xlo=80 * u.deg, xhi=100 * u.deg, zlo=114*u.km/u.s, zhi=314*u.km/u.s)
hipass_cube_K_80_100_p.write('./HIPASS_mosaic_cube_80_100_+.fits', overwrite=True)

hipass_cube_K_80_100_n = hipass_cube_K.subcube(xlo=80 * u.deg, xhi=100 * u.deg, zlo=-258*u.km/u.s, zhi=-58*u.km/u.s)
hipass_cube_K_80_100_n.write('./HIPASS_mosaic_cube_80_100_-.fits', overwrite=True)

hipass_cube_K_100_120_p = hipass_cube_K.subcube(xlo=100 * u.deg, xhi=120 * u.deg, zlo=132*u.km/u.s, zhi=332*u.km/u.s)
hipass_cube_K_100_120_p.write('./HIPASS_mosaic_cube_100_120_+.fits', overwrite=True)

hipass_cube_K_100_120_n = hipass_cube_K.subcube(xlo=100 * u.deg, xhi=120 * u.deg, zlo=-260*u.km/u.s, zhi=-60*u.km/u.s)
hipass_cube_K_100_120_n.write('./HIPASS_mosaic_cube_100_120_-.fits', overwrite=True)

hipass_cube_K_120_140_p = hipass_cube_K.subcube(xlo=120 * u.deg, xhi=140 * u.deg, zlo=139*u.km/u.s, zhi=339*u.km/u.s)
hipass_cube_K_120_140_p.write('./HIPASS_mosaic_cube_120_140_+.fits', overwrite=True)

hipass_cube_K_120_140_n = hipass_cube_K.subcube(xlo=120 * u.deg, xhi=140 * u.deg, zlo=-261*u.km/u.s, zhi=-61*u.km/u.s)
hipass_cube_K_120_140_n.write('./HIPASS_mosaic_cube_120_140_-.fits', overwrite=True)